In [ ]:
# Data Import and Cleaning

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Load the dataset
df = pd.read_csv('global_power_plant_database.csv')

# Display basic information
print("Dataset Shape:", df.shape)
print("\nFirst few rows:")
df.head()

In [ ]:
# Check data types and missing values
print("Data Types:")
print(df.dtypes)
print("\nMissing Values per Column:")
print(df.isnull().sum())

In [ ]:
# Data Cleaning Strategy

# Drop columns with > 60% missing values
missing_percentage = df.isnull().sum() / len(df) * 100
cols_to_drop = missing_percentage[missing_percentage > 60].index.tolist()
print("Columns to drop (>60% missing):", cols_to_drop)

df_clean = df.drop(columns=cols_to_drop)

# Fill missing capacity with median
df_clean['capacity_mw'] = df_clean['capacity_mw'].fillna(df_clean['capacity_mw'].median())

# Fill missing commissioning_year with median
df_clean['commissioning_year'] = df_clean['commissioning_year'].fillna(
    df_clean['commissioning_year'].median()
)

# Drop rows with missing coordinates (needed for geographic analysis)
df_clean = df_clean.dropna(subset=['latitude', 'longitude'])

# Fuel1 has no missing values, so we're good

print(f"\nCleaned dataset shape: {df_clean.shape}")

In [ ]:
# Summary statistics for numerical columns
numerical_cols = ['capacity_mw', 'latitude', 'longitude', 'commissioning_year', 'estimated_generation_gwh']
summary_stats = df_clean[numerical_cols].describe()

print("Summary Statistics:")
summary_stats

In [ ]:
# Additional statistical measures using NumPy
print("=== Advanced Statistical Analysis using NumPy ===\n")

print("Capacity Analysis:")
print(f"  Variance: {np.var(df_clean['capacity_mw']):.2f}")
print(f"  Range: {np.ptp(df_clean['capacity_mw']):.2f}")
print(f"  Skewness: {stats.skew(df_clean['capacity_mw']):.2f}")
print(f"  Kurtosis: {stats.kurtosis(df_clean['capacity_mw']):.2f}")

print("\nCommissioning Year Analysis:")
print(f"  Variance: {np.var(df_clean['commissioning_year']):.2f}")
print(f"  Range: {np.ptp(df_clean['commissioning_year']):.2f}")

print("\nEstimated Generation Analysis:")
print(f"  Variance: {np.var(df_clean['estimated_generation_gwh']):.2f}")
print(f"  Range: {np.ptp(df_clean['estimated_generation_gwh']):.2f}")

In [ ]:
# Top 10 countries by number of power plants
top_countries = df_clean['country'].value_counts().head(10)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Bar plot
top_countries.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Top 10 Countries by Number of Power Plants', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Country', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].tick_params(axis='x', rotation=45)

# Pie chart
colors = plt.cm.Set3(np.linspace(0, 1, 10))
axes[1].pie(top_countries.values, labels=top_countries.index, autopct='%1.1f%%', 
            colors=colors, startangle=90)
axes[1].set_title('Top 10 Countries - Percentage Distribution', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('country_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Fuel type distribution
fuel_distribution = df_clean['fuel1'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Bar plot (top 10 fuel types)
top_fuels = fuel_distribution.head(10)
top_fuels.plot(kind='bar', ax=axes[0], color='coral', edgecolor='black')
axes[0].set_title('Top 10 Fuel Types by Number of Plants', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Fuel Type', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].tick_params(axis='x', rotation=45)

# Pie chart for major fuel categories
major_fuels = fuel_distribution[fuel_distribution > 1000]
others = fuel_distribution[fuel_distribution <= 1000].sum()
major_fuels['Others'] = others

colors = plt.cm.Pastel1(np.linspace(0, 1, len(major_fuels)))
axes[1].pie(major_fuels.values, labels=major_fuels.index, autopct='%1.1f%%', 
            colors=colors, startangle=90)
axes[1].set_title('Fuel Type Distribution (Plants >1000 count grouped)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('fuel_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nFuel Type Summary:")
fuel_distribution.head(15)

In [ ]:
# Calculate statistics for capacity by fuel type
fuel_stats = df_clean.groupby('fuel1')['capacity_mw'].agg(['mean', 'median', 'std', 'count'])
fuel_stats = fuel_stats.sort_values('mean', ascending=False)

print("=== Capacity Statistics by Fuel Type (MW) ===\n")
print(fuel_stats.round(2))

# Generate output for nuclear-specific insights
nuclear_capacity = df_clean[df_clean['fuel1'] == 'Nuclear']['capacity_mw'].sum()
total_capacity = df_clean['capacity_mw'].sum()
print(f"\nNuclear plants represent only {len(df_clean[df_clean['fuel1'] == 'Nuclear'])} plants "
      f"but contribute {nuclear_capacity/total_capacity*100:.1f}% of total capacity.")

In [ ]:
# Function to perform t-test between two fuel types
def compare_fuel_capacities(fuel1, fuel2):
    cap1 = df_clean[df_clean['fuel1'] == fuel1]['capacity_mw'].dropna()
    cap2 = df_clean[df_clean['fuel1'] == fuel2]['capacity_mw'].dropna()
    
    t_stat, p_value = stats.ttest_ind(cap1, cap2)
    
    return {
        'fuel1': fuel1,
        'fuel2': fuel2,
        'mean1': cap1.mean(),
        'mean2': cap2.mean(),
        't_statistic': t_stat,
        'p_value': p_value,
        'significant': p_value < 0.05
    }

# Compare major fuel types
comparisons = [
    ('Nuclear', 'Coal'),
    ('Coal', 'Gas'),
    ('Gas', 'Hydro'),
    ('Hydro', 'Wind'),
    ('Coal', 'Solar')
]

print("=== Hypothesis Testing Results ===\n")
for f1, f2 in comparisons:
    result = compare_fuel_capacities(f1, f2)
    print(f"{f1} vs {f2}:")
    print(f"  Mean capacities: {result['mean1']:.1f} MW vs {result['mean2']:.1f} MW")
    print(f"  T-statistic: {result['t_statistic']:.3f}")
    print(f"  P-value: {result['p_value']:.6f}")
    print(f"  Significant difference: {result['significant']}\n")

In [ ]:
# Filter data from 1950 onwards for better visualization
time_df = df_clean[df_clean['commissioning_year'] >= 1950].copy()

# Create decade bins
time_df['decade'] = (time_df['commissioning_year'] // 10) * 10

# Calculate fuel mix by decade
fuel_by_decade = time_df.groupby(['decade', 'fuel1']).size().unstack(fill_value=0)

# Normalize to percentages
fuel_by_decade_pct = fuel_by_decade.div(fuel_by_decade.sum(axis=1), axis=0) * 100

# Select top fuel types for clarity
top_fuels = fuel_by_decade.sum().sort_values(ascending=False).head(8).index
fuel_by_decade_pct_filtered = fuel_by_decade_pct[top_fuels]

# Plot
fig, ax = plt.subplots(figsize=(14, 8))

fuel_by_decade_pct_filtered.plot(kind='area', stacked=True, ax=ax, alpha=0.8)
ax.set_title('Evolution of Global Power Plant Fuel Mix by Decade', fontsize=16, fontweight='bold')
ax.set_xlabel('Decade', fontsize=12)
ax.set_ylabel('Percentage of Plants (%)', fontsize=12)
ax.legend(title='Fuel Type', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('fuel_mix_evolution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Analyze average capacity trends by fuel type
fig, ax = plt.subplots(figsize=(14, 8))

# Calculate average capacity by decade for each fuel type
for fuel in ['Nuclear', 'Coal', 'Gas', 'Hydro', 'Wind', 'Solar']:
    fuel_data = time_df[time_df['fuel1'] == fuel]
    if len(fuel_data) > 10:
        avg_capacity = fuel_data.groupby('decade')['capacity_mw'].mean()
        ax.plot(avg_capacity.index, avg_capacity.values, marker='o', linewidth=2, label=fuel)

ax.set_title('Average Plant Capacity Trends by Fuel Type', fontsize=16, fontweight='bold')
ax.set_xlabel('Decade', fontsize=12)
ax.set_ylabel('Average Capacity (MW)', fontsize=12)
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
ax.set_yscale('log')  # Log scale for better visualization

plt.tight_layout()
plt.savefig('capacity_trends.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Create a world map of power plants by fuel type
fig, ax = plt.subplots(figsize=(16, 10))

# Define colors for major fuel types
fuel_colors = {
    'Hydro': 'blue',
    'Gas': 'red',
    'Oil': 'orange',
    'Coal': 'black',
    'Wind': 'green',
    'Solar': 'yellow',
    'Nuclear': 'purple',
    'Biomass': 'brown'
}

# Plot each fuel type
for fuel, color in fuel_colors.items():
    fuel_data = df_clean[df_clean['fuel1'] == fuel]
    if len(fuel_data) > 0:
        ax.scatter(fuel_data['longitude'], fuel_data['latitude'], 
                  c=color, label=fuel, alpha=0.5, s=10)

ax.set_title('Global Distribution of Power Plants by Fuel Type', fontsize=16, fontweight='bold')
ax.set_xlabel('Longitude', fontsize=12)
ax.set_ylabel('Latitude', fontsize=12)
ax.legend(loc='upper right', ncol=2)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('global_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Create correlation matrix for numerical variables
corr_matrix = df_clean[['capacity_mw', 'latitude', 'longitude', 
                        'commissioning_year', 'estimated_generation_gwh']].corr()

print("=== Correlation Matrix ===")
print(corr_matrix.round(3))

# Visualize correlation matrix
fig, ax = plt.subplots(figsize=(10, 8))

# Using NumPy for array operations on correlation matrix
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.3f', cmap='coolwarm',
            center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8})

ax.set_title('Correlation Matrix of Power Plant Attributes', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Prepare data for eigen analysis
features_for_pca = ['capacity_mw', 'latitude', 'longitude', 'commissioning_year']
X = df_clean[features_for_pca].dropna().values

# Standardize the data (important for PCA)
X_centered = X - np.mean(X, axis=0)
X_scaled = X_centered / np.std(X, axis=0)

# Compute covariance matrix
cov_matrix = np.cov(X_scaled.T)

# Calculate eigenvalues and eigenvectors
eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)

# Sort eigenvalues in descending order
idx = eigenvalues.argsort()[::-1]
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

print("=== Eigenvalues and Explained Variance ===\n")
print("Eigenvalues:", eigenvalues.round(4))

# Calculate explained variance ratio
explained_variance_ratio = eigenvalues / np.sum(eigenvalues)
cumulative_variance = np.cumsum(explained_variance_ratio)

for i, (ev, evr) in enumerate(zip(eigenvalues, explained_variance_ratio)):
    print(f"PC{i+1}: Eigenvalue={ev:.4f}, "
          f"Explained Variance={evr*100:.2f}%, "
          f"Cumulative={cumulative_variance[i]*100:.2f}%")

print("\n=== Principal Component Loadings (Eigenvectors) ===\n")
for i in range(len(features_for_pca)):
    print(f"PC{i+1}: {', '.join([f'{features_for_pca[j]}: {eigenvectors[j,i]:.4f}' for j in range(len(features_for_pca))])}")

In [ ]:
# Example: Using matrix operations to find similar plants
def find_similar_plants(plant_index, n_neighbors=5):
    """Find plants similar to a given plant using Euclidean distance"""
    features = ['capacity_mw', 'commissioning_year', 'latitude', 'longitude']
    plant_features = df_clean[features].dropna()
    
    # Standardize features using NumPy
    X = plant_features.values
    X_mean = np.mean(X, axis=0)
    X_std = np.std(X, axis=0)
    X_scaled = (X - X_mean) / X_std
    
    # Calculate distance matrix using broadcasting
    target = X_scaled[plant_index]
    distances = np.sqrt(np.sum((X_scaled - target) ** 2, axis=1))
    
    # Get nearest neighbors
    nearest_indices = np.argsort(distances)[1:n_neighbors+1]
    
    return plant_features.iloc[nearest_indices]

# Find plants similar to the largest capacity plant
largest_plant_idx = df_clean['capacity_mw'].idxmax()
largest_plant = df_clean.loc[largest_plant_idx]

print(f"Largest plant: {largest_plant['name']}")
print(f"  Country: {largest_plant['country']}")
print(f"  Capacity: {largest_plant['capacity_mw']} MW")
print(f"  Fuel: {largest_plant['fuel1']}")

similar = find_similar_plants(df_clean.index.get_loc(largest_plant_idx))
print("\nSimilar plants (by capacity, year, and location):")
similar.head()

In [ ]:
# Using NumPy for complex filtering conditions
# Find large capacity (>1000 MW) plants that are also in the top 10% by estimated generation
capacity_array = df_clean['capacity_mw'].values
generation_array = df_clean['estimated_generation_gwh'].values

# Calculate threshold for top 10% generation (using NumPy percentile)
gen_threshold = np.percentile(generation_array[~np.isnan(generation_array)], 90)

# Create boolean mask using NumPy conditions
mask = (capacity_array > 1000) & (generation_array > gen_threshold)

# Apply mask to DataFrame
large_efficient_plants = df_clean[mask]

print(f"Plants with >1000 MW capacity and top 10% generation: {len(large_efficient_plants)}")
print("\nTop 5 such plants:")
large_efficient_plants.nlargest(5, 'estimated_generation_gwh')[['name', 'country', 'fuel1', 'capacity_mw', 'estimated_generation_gwh']]

In [ ]:
# Create a sophisticated visualization using NumPy for data preparation
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# 1. Distribution of capacities by fuel type (using NumPy array operations)
fuel_types = ['Coal', 'Gas', 'Hydro', 'Nuclear', 'Wind', 'Solar']
capacity_data = []

for fuel in fuel_types:
    capacities = df_clean[df_clean['fuel1'] == fuel]['capacity_mw'].values
    # Apply log transformation using NumPy
    log_capacities = np.log1p(capacities)
    capacity_data.append(log_capacities)

# Create violin plot
parts = ax1.violinplot(capacity_data, positions=range(len(fuel_types)), 
                       showmeans=True, showmedians=True)

ax1.set_xticks(range(len(fuel_types)))
ax1.set_xticklabels(fuel_types, rotation=45)
ax1.set_ylabel('log(Capacity + 1) [MW]', fontsize=12)
ax1.set_title('Capacity Distribution by Fuel Type (Log Scale)', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)

# 2. Scatter plot with trend lines using NumPy polyfit
for fuel, color in zip(['Coal', 'Gas', 'Hydro', 'Wind'], ['black', 'red', 'blue', 'green']):
    fuel_data = df_clean[df_clean['fuel1'] == fuel].dropna(subset=['commissioning_year', 'capacity_mw'])
    years = fuel_data['commissioning_year'].values
    capacities = fuel_data['capacity_mw'].values
    
    # Fit polynomial using NumPy
    z = np.polyfit(years[years >= 1970], capacities[years >= 1970], 1)
    p = np.poly1d(z)
    
    ax2